In [ ]:
pip install openmeteo-requests requests-cache retry-requests

In [3]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": 3.1412,
	"longitude": 101.6865,
	"hourly": ["temperature_2m", "rain", "weather_code", "cloud_cover", "visibility", "wind_speed_10m", "precipitation", "precipitation_probability", "apparent_temperature"],
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_rain = hourly.Variables(1).ValuesAsNumpy()
hourly_weather_code = hourly.Variables(2).ValuesAsNumpy()
hourly_cloud_cover = hourly.Variables(3).ValuesAsNumpy()
hourly_visibility = hourly.Variables(4).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(5).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(6).ValuesAsNumpy()
hourly_precipitation_probability = hourly.Variables(7).ValuesAsNumpy()
hourly_apparent_temperature = hourly.Variables(8).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
	end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["rain"] = hourly_rain
hourly_data["weather_code"] = hourly_weather_code
hourly_data["cloud_cover"] = hourly_cloud_cover
hourly_data["visibility"] = hourly_visibility
hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
hourly_data["precipitation"] = hourly_precipitation
hourly_data["precipitation_probability"] = hourly_precipitation_probability
hourly_data["apparent_temperature"] = hourly_apparent_temperature

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)

Coordinates: 3.125°N 101.625°E
Elevation: 47.0 m asl
Timezone difference to GMT+0: 0s

Hourly data
                          date  temperature_2m  rain  weather_code  \
0   2026-04-13 00:00:00+00:00       27.175501   0.0           3.0   
1   2026-04-13 01:00:00+00:00       29.125502   0.0           2.0   
2   2026-04-13 02:00:00+00:00       30.625502   0.0           3.0   
3   2026-04-13 03:00:00+00:00       31.725500   0.0           2.0   
4   2026-04-13 04:00:00+00:00       32.675499   0.0           2.0   
..                        ...             ...   ...           ...   
163 2026-04-19 19:00:00+00:00       26.725500   0.0           3.0   
164 2026-04-19 20:00:00+00:00       26.375502   0.0           3.0   
165 2026-04-19 21:00:00+00:00       26.075500   0.0           3.0   
166 2026-04-19 22:00:00+00:00       25.775501   0.0          95.0   
167 2026-04-19 23:00:00+00:00       25.525501   0.0          95.0   

     cloud_cover  visibility  wind_speed_10m  precipitation  \
0       

In [ ]:

hourly_dataframe.info()
hourly_dataframe.describe()


<class 'pandas.DataFrame'>
RangeIndex: 168 entries, 0 to 167
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype             
---  ------                     --------------  -----             
 0   date                       168 non-null    datetime64[s, UTC]
 1   temperature_2m             168 non-null    float32           
 2   rain                       168 non-null    float32           
 3   weather_code               168 non-null    float32           
 4   cloud_cover                168 non-null    float32           
 5   visibility                 168 non-null    float32           
 6   wind_speed_10m             168 non-null    float32           
 7   precipitation              168 non-null    float32           
 8   precipitation_probability  168 non-null    float32           
 9   apparent_temperature       168 non-null    float32           
dtypes: datetime64[s, UTC](1), float32(9)
memory usage: 7.3 KB


,temperature_2m,rain,weather_code,cloud_cover,visibility,wind_speed_10m,precipitation,precipitation_probability,apparent_temperature
count,168.000000,168.0,168.000000,168.000000,168.000000,168.000000,168.000000,168.000000,168.000000
mean,28.453474,0.0,30.351191,97.380951,20454.166016,4.108024,0.519048,43.898811,34.233974
std,2.493883,0.0,40.184772,8.682566,6012.041504,2.807945,0.946785,24.263681,2.626882
min,25.025501,0.0,2.000000,48.000000,3240.000000,0.360000,0.000000,0.000000,30.142807
25%,26.313001,0.0,3.000000,100.000000,19200.000000,1.912906,0.000000,25.000000,32.047854
50%,27.775500,0.0,3.000000,100.000000,24140.000000,3.279518,0.000000,40.500000,33.709555
75%,30.450501,0.0,80.000000,100.000000,24140.000000,5.937272,0.625000,66.000000,36.178364
max,34.625500,0.0,96.000000,100.000000,24140.000000,13.556282,5.000000,95.000000,41.128555


In [7]:
hourly_dataframe.head()

,date,temperature_2m,rain,weather_code,cloud_cover,visibility,wind_speed_10m,precipitation,precipitation_probability,apparent_temperature
0,2026-04-13 00:00:00+00:00,27.175501,0.0,3.0,57.0,24140.0,1.800000,0.0,15.0,33.659496
1,2026-04-13 01:00:00+00:00,29.125502,0.0,2.0,61.0,22960.0,1.527351,0.0,8.0,35.708511
2,2026-04-13 02:00:00+00:00,30.625502,0.0,3.0,89.0,24140.0,3.075841,0.2,23.0,36.908020
3,2026-04-13 03:00:00+00:00,31.725500,0.0,2.0,51.0,24140.0,5.154415,0.1,38.0,38.201450
4,2026-04-13 04:00:00+00:00,32.675499,0.0,2.0,68.0,22860.0,4.334974,0.1,40.0,39.398682
